In [ ]:
from __future__ import annotations
import os
import subprocess
import sys
import time
from pathlib import Path
from dotenv import load_dotenv
_ = load_dotenv()

CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()
if (
    (CURRENT_WORKING_DIRECTORY / "main.ipynb").exists()
    and (CURRENT_WORKING_DIRECTORY / "standalone_scripts").exists()
):
    PROJECT_ROOT = CURRENT_WORKING_DIRECTORY
elif (
    CURRENT_WORKING_DIRECTORY.name == "standalone_scripts"
    and (CURRENT_WORKING_DIRECTORY.parent / "main.ipynb").exists()
):
    PROJECT_ROOT = CURRENT_WORKING_DIRECTORY.parent
else:
    raise RuntimeError(
        f"Unable to determine project root from cwd={CURRENT_WORKING_DIRECTORY}"
    )

SCRIPTS_DIR = PROJECT_ROOT / "standalone_scripts"
MAIN_EXTRACTOR_PATH = SCRIPTS_DIR / "main_extractor.py"
DATE_EXPANDER_PATH = SCRIPTS_DIR / "date_expander.py"

VENV_PYTHON_PATH = PROJECT_ROOT / ".venv" / "bin" / "python"
PYTHON_RUNNER = str(VENV_PYTHON_PATH if VENV_PYTHON_PATH.exists() else Path(sys.executable))

ACTIVE_STARTING_PAGE_URLS = os.environ.get("ACTIVE_STARTING_PAGE_URLS")


def run_command(command_parts: list[str], label: str) -> int:
    print(f"\n[{label}] Running command:")
    print(" ".join(command_parts))

    started_at = time.perf_counter()
    completed_process = subprocess.run(
        command_parts,
        cwd=str(PROJECT_ROOT),
        check=False,
    )
    elapsed_seconds = time.perf_counter() - started_at
    print(
        f"[{label}] Exit code={completed_process.returncode} elapsed={elapsed_seconds:.1f}s"
    )
    return completed_process.returncode


if not MAIN_EXTRACTOR_PATH.exists():
    raise FileNotFoundError(f"Missing script: {MAIN_EXTRACTOR_PATH}")
if not DATE_EXPANDER_PATH.exists():
    raise FileNotFoundError(f"Missing script: {DATE_EXPANDER_PATH}")

print(f"[orchestrator] Active URLs: {len(ACTIVE_STARTING_PAGE_URLS)}")

url_results: list[dict[str, object]] = []
for url_index, starting_page_url in enumerate(ACTIVE_STARTING_PAGE_URLS, start=1):
    url_label = f"URL {url_index}/{len(ACTIVE_STARTING_PAGE_URLS)}"
    print(f"\n[orchestrator] {url_label} -> {starting_page_url}")

    command_parts = [
        PYTHON_RUNNER,
        str(MAIN_EXTRACTOR_PATH),
        "--starting-page-url",
        starting_page_url,
    ]
    return_code = run_command(command_parts=command_parts, label=url_label)

    url_results.append(
        {
            "url": starting_page_url,
            "return_code": return_code,
            "status": "ok" if return_code == 0 else "failed",
        }
    )

print("\n[orchestrator] URL pipeline summary")
for result in url_results:
    print(f"- {result['status']}: {result['url']} (code={result['return_code']})")

failed_count = sum(1 for result in url_results if result["return_code"] != 0)
print(
    f"[orchestrator] URL run complete. total={len(url_results)} failed={failed_count}"
)

print("\n[orchestrator] Running date expansion apply step")
date_expander_command_parts = [
    PYTHON_RUNNER,
    str(DATE_EXPANDER_PATH),
    "--mode",
    "apply",
    "--limit",
    "500",
]
date_expander_return_code = run_command(
    command_parts=date_expander_command_parts,
    label="date_expander",
)

if date_expander_return_code != 0:
    raise RuntimeError(
        f"date_expander.py failed with return code {date_expander_return_code}"
    )

print("\n[orchestrator] Done")